# Chapter 4: Convolutional Neural Networks

*Deep Learning Crash Course - BPB Publications*

This notebook implements the chapter's convolutional models on CIFAR-10, demonstrates residual connections and batch normalisation, plots feature maps and Grad-CAM, and concludes with the chapter exercises.


## 1. Setup

In [ ]:
import os
# CPU-mode guard: hide the GPU from TensorFlow before it is imported.
# Many machines have an NVIDIA driver but a missing CUDA toolkit
# (no `ptxas`), in which case TF's XLA JIT crashes the kernel. Remove
# these two lines once you have a complete CUDA install.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

IMG_DIR = Path('images'); IMG_DIR.mkdir(exist_ok=True)
def set_seed(s=42):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    try:
        import tensorflow as tf; tf.random.set_seed(s)
    except ModuleNotFoundError: pass
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ModuleNotFoundError: pass
set_seed(42)


## 2. Convolution from scratch in NumPy

Understanding the operation before letting a framework do it.

In [ ]:
def conv2d_naive(image, kernel, stride=1, padding=0):
    '''Single-channel 2D convolution (cross-correlation) with NumPy.'''
    if padding:
        image = np.pad(image, ((padding, padding), (padding, padding)))
    H, W = image.shape
    kH, kW = kernel.shape
    oH = (H - kH) // stride + 1
    oW = (W - kW) // stride + 1
    out = np.zeros((oH, oW), dtype=image.dtype)
    for i in range(oH):
        for j in range(oW):
            patch = image[i*stride:i*stride+kH, j*stride:j*stride+kW]
            out[i, j] = (patch * kernel).sum()
    return out

image = np.array([[1, 2, 3, 4, 5],
                  [6, 7, 8, 9, 10],
                  [11, 12, 13, 14, 15],
                  [16, 17, 18, 19, 20],
                  [21, 22, 23, 24, 25]], dtype=float)
sobel = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=float)
print('Conv output:')
print(conv2d_naive(image, sobel, stride=1, padding=0))


## 3. Output dimensions, stride, padding

The output spatial dimension is $(W - K + 2P) / S + 1$. The cell verifies that formula for many configurations.

In [ ]:
def output_size(W, K, P, S):
    return (W - K + 2 * P) // S + 1

rows = []
for W in [28, 32, 64]:
    for K, P, S in [(3, 0, 1), (3, 1, 1), (5, 2, 1), (3, 0, 2), (7, 3, 2)]:
        rows.append(((W, K, P, S), output_size(W, K, P, S)))
for cfg, o in rows:
    print(f'W={cfg[0]} K={cfg[1]} P={cfg[2]} S={cfg[3]} -> {o}')


## 4. CIFAR-10 baseline CNN in Keras

In [ ]:
import tensorflow as tf
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
X_train, X_test = X_train.astype('float32') / 255.0, X_test.astype('float32') / 255.0
y_train_oh = tf.keras.utils.to_categorical(y_train, 10)
y_test_oh = tf.keras.utils.to_categorical(y_test, 10)
CIFAR_CLASSES = ['airplane', 'auto', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, img, lab in zip(axes.flat, X_train[:10], y_train[:10]):
    ax.imshow(img); ax.set_title(CIFAR_CLASSES[int(lab)]); ax.axis('off')
fig.suptitle('CIFAR-10 sample images')
fig.tight_layout(); fig.savefig(IMG_DIR / '01_cifar_samples.png', dpi=150); plt.show()


In [ ]:
def baseline_cnn():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])

set_seed(42)
model = baseline_cnn()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
history = model.fit(X_train, y_train_oh, validation_split=0.1, epochs=6, batch_size=128, verbose=2)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.history['accuracy'], label='train acc'); ax.plot(history.history['val_accuracy'], label='val acc')
ax.set_xlabel('epoch'); ax.set_ylabel('accuracy'); ax.set_title('CIFAR-10 baseline CNN')
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(IMG_DIR / '02_baseline_cnn.png', dpi=150); plt.show()
print('Test:', model.evaluate(X_test, y_test_oh, verbose=0))


## 5. Residual block + BatchNorm + augmentation

In [ ]:
from tensorflow.keras import layers, Model, Input

def conv_bn_relu(x, filters, kernel_size=3, stride=1):
    x = layers.Conv2D(filters, kernel_size, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def residual_block(x, filters, stride=1):
    shortcut = x
    x = conv_bn_relu(x, filters, 3, stride)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    return layers.ReLU()(layers.Add()([x, shortcut]))

def mini_resnet():
    inp = Input((32, 32, 3))
    x = conv_bn_relu(inp, 32, 3)
    x = residual_block(x, 32)
    x = residual_block(x, 64, stride=2)
    x = residual_block(x, 128, stride=2)
    x = layers.GlobalAveragePooling2D()(x)
    out = layers.Dense(10, activation='softmax')(x)
    return Model(inp, out)

set_seed(42)
augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
])
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train_oh))
train_ds = train_ds.shuffle(2048).batch(128).map(lambda x, y: (augment(x, training=True), y),
                                                  num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test_oh)).batch(256)

res_model = mini_resnet()
res_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
h2 = res_model.fit(train_ds, validation_data=val_ds, epochs=6, verbose=2)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.history['val_accuracy'], label='baseline CNN val')
ax.plot(h2.history['val_accuracy'], label='mini ResNet+aug val')
ax.set_xlabel('epoch'); ax.set_ylabel('val accuracy'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('CIFAR-10: baseline vs mini ResNet')
fig.tight_layout(); fig.savefig(IMG_DIR / '03_resnet_compare.png', dpi=150); plt.show()


## 6. Visualising what the network learns: feature maps + Grad-CAM

In [ ]:
# Pick a single test image and visualise the first conv-block activations
sample = X_test[5:6]
conv_layer_names = [l.name for l in res_model.layers if 'conv' in l.name][:4]
feature_model = tf.keras.Model(inputs=res_model.input,
                               outputs=[res_model.get_layer(name).output for name in conv_layer_names])
feats = feature_model(sample)
fig, axes = plt.subplots(len(conv_layer_names), 6, figsize=(12, 2 * len(conv_layer_names)))
for row, (name, f) in enumerate(zip(conv_layer_names, feats)):
    for col in range(6):
        axes[row, col].imshow(f[0, :, :, col], cmap='viridis')
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name, rotation=0, labelpad=40, fontsize=9)
fig.suptitle('Feature maps across early conv layers'); fig.tight_layout()
fig.savefig(IMG_DIR / '04_feature_maps.png', dpi=150); plt.show()


In [ ]:
def gradcam(model, image, class_index, last_conv_name):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(image)
        loss = preds[:, class_index]
    grads = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_out[0] * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

last_conv = [l.name for l in res_model.layers if isinstance(l, tf.keras.layers.Conv2D)][-1]
fig, axes = plt.subplots(2, 4, figsize=(11, 5))
for i, ax_pair in enumerate(axes.T):
    img = X_test[i + 10]
    pred = res_model.predict(img[None], verbose=0)
    cls = int(np.argmax(pred))
    cam = gradcam(res_model, img[None], cls, last_conv)
    ax_pair[0].imshow(img); ax_pair[0].set_title(CIFAR_CLASSES[cls]); ax_pair[0].axis('off')
    ax_pair[1].imshow(img); ax_pair[1].imshow(np.array(plt.cm.jet(cam)[..., :3]),
                                              alpha=0.45, extent=(0, 32, 32, 0))
    ax_pair[1].set_title('Grad-CAM'); ax_pair[1].axis('off')
fig.suptitle('Grad-CAM heatmaps over a few CIFAR-10 test samples')
fig.tight_layout(); fig.savefig(IMG_DIR / '05_gradcam.png', dpi=150); plt.show()


## 7. Transfer learning preview - feature extraction with MobileNetV2

Full transfer learning is the subject of Chapter 8, but freezing a pretrained backbone and adding a small head is a one-liner in Keras.

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights='imagenet')
base.trainable = False
ft = tf.keras.Sequential([
    tf.keras.layers.Resizing(96, 96),
    tf.keras.layers.Rescaling(scale=2.0, offset=-1.0),
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(10, activation='softmax'),
])
ft.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
h_ft = ft.fit(X_train[:5000], y_train_oh[:5000], validation_data=(X_test[:1000], y_test_oh[:1000]),
              epochs=3, batch_size=128, verbose=2)


## 8. Exercise solutions

### 8.1 MCQ answer key

| Q | Answer | Why |
|---|--------|-----|
| 1 | (b) Parameter sharing exploits translational structure | Same filter slides across the image; massively fewer parameters than a dense layer. |
| 2 | (b) 26x26 | $(28 - 3 + 0) / 1 + 1 = 26$. |
| 3 | (b) Max-pooling | Mean / average is the alternative. |
| 4 | (c) Degradation problem | Deeper plain networks were harder to optimise than shallower ones until skip connections. |
| 5 | (c) The block is the identity map | $H(x) = 0 + x$. |
| 6 | (b) After Conv, before ReLU | Conv $\rightarrow$ BN $\rightarrow$ ReLU is the canonical ordering. |
| 7 | (c) Horizontal flip | Flipping a 'no left turn' sign inverts its meaning. |
| 8 | (a) All pretrained backbone weights | Only the new classification head trains. |
| 9 | (b) Pretrained weights are well-calibrated | Large updates destroy useful features. |
| 10 | (c) Flatten + Dense head | GAP keeps spatial information per channel without learned weights. |
| 11 | (b) Class-discriminative spatial regions | Where the CNN looked to make its decision. |
| 12 | (b) Depth, width and resolution | Compound scaling sweeps all three with one coefficient. |


### 8.2 Why convolutions beat fully connected layers on images
Three reasons:
1. **Parameter sharing** - one set of weights per filter scans the whole image. A $32 \times 32 \times 3$ image into 64 hidden units is $\sim 200k$ weights for a dense layer but only $1.7k$ for a $3 \times 3 \times 3 \rightarrow 64$ conv.
2. **Translation equivariance** - moving the cat 5 pixels to the right moves the response 5 pixels to the right but does not change its content.
3. **Locality** - small receptive fields force the network to first learn local edges and textures before composing them into objects.

### 8.3 Output-size arithmetic
Already verified in section 3. The general formula is
$$o = \lfloor (i - k + 2p) / s \rfloor + 1$$

### 8.4 Residual connections and the degradation problem
Plain stacks of conv layers showed surprisingly *higher training error* as depth grew past 20 layers - the network had trouble even fitting the training set. He et al. showed that learning a residual $F(x)$ on top of the identity $x$ is much easier than learning $H(x)$ from scratch: if the optimal mapping is close to the identity, $F(x) \approx 0$ is reachable, but learning $H(x) \approx x$ in a stack of nonlinear layers requires precise weight cooperation.

### 8.5 BatchNorm placement
The canonical order is **Conv -> BatchNorm -> Activation**. BatchNorm before the non-linearity ensures the input to the non-linearity has roughly zero mean and unit variance, which keeps gradients in the well-behaved part of ReLU. The BatchNorm-after-ReLU ordering exists in some papers but is less common.

### 8.6 Data augmentation for a traffic-sign classifier
Safe augmentations: small translation, brightness, additive noise.
**Unsafe**: horizontal flip - it inverts meaning ('no left turn' to 'no right turn'); large rotations - signs are upright.


### 8.7 Transfer learning - what is frozen?
In feature-extraction mode every backbone weight is frozen (`trainable=False` in Keras, `requires_grad=False` in PyTorch). Only the new classification head learns. BatchNorm running statistics must also stay in inference mode - that is what `training=False` in the Keras call achieves.

### 8.8 Lower learning rate during fine-tuning
Pretrained backbone weights already represent useful features. A large LR will move them far away in the first few steps before the head has converged, destroying the very features that made transfer useful. Typical recipe: head LR $10^{-3}$, backbone LR $10^{-5}$ once unfrozen.

### 8.9 Global Average Pooling vs Flatten + Dense
GAP reduces a $(H, W, C)$ feature map to a $C$-vector by averaging over space. It eliminates the huge dense layer that Flatten otherwise feeds, sharply reducing parameter count and over-fitting risk, and it is permutation-invariant in space.

### 8.10 Grad-CAM
Grad-CAM highlights the **class-discriminative spatial regions** the CNN attended to. It computes the gradient of the target class score with respect to the last convolutional feature map, averages those gradients per channel to obtain weights, and combines them into a heatmap. Section 6 above shows it in action.

### 8.11 EfficientNet compound scaling
EfficientNet's scaling coefficients $\phi$ multiply depth, width and resolution simultaneously: $\text{depth} = \alpha^\phi, \text{width} = \beta^\phi, \text{resolution} = \gamma^\phi$, subject to $\alpha \beta^2 \gamma^2 \approx 2$. Scaling all three together gives better accuracy at a fixed FLOPs budget than scaling any one dimension alone.

---
*End of Chapter 4.*
